### Reading in the raw csv file to stream into the database

In [ ]:
import pandas as pd

In [390]:
patient_visits = pd.read_csv(r"D:\Google Drive\Data Analysis Education\Portfolio Projects\Diagnosing the Data An Analysis of Patient Demographics, Common Conditions, and Care Utilization\Data\patient_visits.csv")
icd_codes = pd.read_csv(r"D:\Google Drive\Data Analysis Education\Portfolio Projects\Diagnosing the Data An Analysis of Patient Demographics, Common Conditions, and Care Utilization\Data\icd_codes.csv")


### Viewing the patient visit data

In [391]:
patient_visits

,visit_id,patient_id,visit_date,date_of_birth,patient_age,patient_sex,icd_code,cpt_code
0,V000001,PT00001,3/15/2022,7/22/1956,66,Male,I10,12345
1,V000002,PT00002,4/22/2022,11/5/1982,39,Female,E11.9,23456
2,V000003,PT00003,5/10/2022,2/14/1978,44,Male,J45.909,34567
3,V000004,PT00004,1/18/2023,9/2/2004,57,Female,N39.0,45678
4,V000005,PT00005,7/30/2023,3/19/1948,75,Male,I25.110,56789
...,...,...,...,...,...,...,...,...
195,V000018,PT00018,9/9/2024,12/3/1977,46,Female,I10,98011
196,V000019,PT00019,5/25/2022,7/21/1995,28,Male,J02.9,21004
197,V000020,PT00020,7/12/2023,11/2/2004,38,Female,E78.0,26009
198,V000021,PT00021,4/3/2024,4/8/1969,55,Male,I10,52018


### Let's check some info about the data

In [392]:
patient_visits.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   visit_id       200 non-null    str  
 1   patient_id     200 non-null    str  
 2   visit_date     200 non-null    str  
 3   date_of_birth  200 non-null    str  
 4   patient_age    200 non-null    int64
 5   patient_sex    200 non-null    str  
 6   icd_code       200 non-null    str  
 7   cpt_code       200 non-null    int64
dtypes: int64(2), str(6)
memory usage: 12.6 KB


In [393]:
patient_visits.shape

(200, 8)

In [394]:
patient_visits.dtypes

visit_id           str
patient_id         str
visit_date         str
date_of_birth      str
patient_age      int64
patient_sex        str
icd_code           str
cpt_code         int64
dtype: object

### Let's also check unique rows in the data

In [395]:
patient_visits.nunique()

visit_id          42
patient_id        42
visit_date       157
date_of_birth    187
patient_age       54
patient_sex        2
icd_code          55
cpt_code         129
dtype: int64

### We'll start by fixing the data types of the date columns

In [396]:
patient_visits['date_of_birth'] = pd.to_datetime(patient_visits['date_of_birth'])
patient_visits['visit_date'] = pd.to_datetime(patient_visits['visit_date'])

### Also, convert the CPT column to string

In [397]:
patient_visits['cpt_code'] = patient_visits['cpt_code'].astype(str)

In [398]:
patient_visits.head()

,visit_id,patient_id,visit_date,date_of_birth,patient_age,patient_sex,icd_code,cpt_code
0,V000001,PT00001,2022-03-15,1956-07-22,66,Male,I10,12345
1,V000002,PT00002,2022-04-22,1982-11-05,39,Female,E11.9,23456
2,V000003,PT00003,2022-05-10,1978-02-14,44,Male,J45.909,34567
3,V000004,PT00004,2023-01-18,2004-09-02,57,Female,N39.0,45678
4,V000005,PT00005,2023-07-30,1948-03-19,75,Male,I25.110,56789


### Do the Visit dates and DOB seem plausible? It seems so.

In [399]:
patient_visits['date_of_birth'].min()


Timestamp('1938-12-01 00:00:00')

In [400]:
patient_visits['date_of_birth'].max()

Timestamp('2006-01-05 00:00:00')

In [401]:
patient_visits['visit_date'].min()

Timestamp('2022-01-09 00:00:00')

In [402]:
patient_visits['visit_date'].max()

Timestamp('2024-12-18 00:00:00')

### We will remove duplicates using 3 columns and keep the first row and drop the rest of the rows to determine the number of patients

In [403]:
patients = patient_visits.drop_duplicates(subset= 'patient_id',keep = 'first')[['patient_id','date_of_birth','patient_sex']]

patients

,patient_id,date_of_birth,patient_sex
0,PT00001,1956-07-22,Male
1,PT00002,1982-11-05,Female
2,PT00003,1978-02-14,Male
3,PT00004,2004-09-02,Female
4,PT00005,1948-03-19,Male
5,PT00006,1988-06-21,Female
6,PT00007,1992-08-15,Female
7,PT00008,1970-12-01,Male
8,PT00009,1960-04-28,Female
9,PT00010,1985-05-30,Male


In [404]:
patients.shape


(42, 3)

In [405]:
patients['patient_id'].nunique()

42

### Let's calculate the age using the DOB

In [406]:
today = pd.to_datetime('2024-12-18')

patients['dob_age'] = today.year - patients['date_of_birth'].dt.year

In [407]:
patients

,patient_id,date_of_birth,patient_sex,dob_age
0,PT00001,1956-07-22,Male,68
1,PT00002,1982-11-05,Female,42
2,PT00003,1978-02-14,Male,46
3,PT00004,2004-09-02,Female,20
4,PT00005,1948-03-19,Male,76
5,PT00006,1988-06-21,Female,36
6,PT00007,1992-08-15,Female,32
7,PT00008,1970-12-01,Male,54
8,PT00009,1960-04-28,Female,64
9,PT00010,1985-05-30,Male,39


### Let's group patients into age brackets of 0–17, 18–39, 40–64, 65+

In [408]:
def age_brackets(dob_age):
    if dob_age>= 65:
        return '65+'
    elif dob_age>= 40:
        return '40-64'
    elif dob_age>= 18:
        return '18-39'
    else:
        return '0-17'


patients['age_bracket'] = patients['dob_age'].apply(age_brackets)

patients.sort_values(by = 'dob_age')

,patient_id,date_of_birth,patient_sex,dob_age,age_bracket
3,PT00004,2004-09-02,Female,20,18-39
143,PT10006,2001-07-08,Female,23,18-39
149,PT10012,1995-09-03,Male,29,18-39
17,PT00018,1994-06-17,Female,30,18-39
6,PT00007,1992-08-15,Female,32,18-39
151,PT10014,1992-06-14,Male,32,18-39
155,PT10018,1991-12-29,Male,33,18-39
199,PT00022,1991-02-25,Female,33,18-39
11,PT00012,1990-09-25,Male,34,18-39
144,PT10007,1988-12-22,Female,36,18-39


In [409]:
agg_age_bracket = patients.groupby('age_bracket')['patient_id'].count().to_frame(name = 'no_of_patients').reset_index()

agg_age_bracket

,age_bracket,no_of_patients
0,18-39,15
1,40-64,20
2,65+,7


In [410]:
agg_age_bracket.to_csv(r"D:\Google Drive\Data Analysis Education\Portfolio Projects\Diagnosing the Data An Analysis of Patient Demographics, Common Conditions, and Care Utilization\Exports\CSV\age_bracket_summary.csv",index=False)

### Moving on to the icd_codes data

In [413]:
icd_codes.head()

,Code,Short Description
0,A000,"Cholera due to Vibrio cholerae 01, biovar chol..."
1,A001,"Cholera due to Vibrio cholerae 01, biovar eltor"
2,A009,"Cholera, unspecified"
3,A0100,"Typhoid fever, unspecified"
4,A0101,Typhoid meningitis


In [414]:
icd_codes.info()

<class 'pandas.DataFrame'>
RangeIndex: 73895 entries, 0 to 73894
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Code               73895 non-null  str  
 1   Short Description  73895 non-null  str  
dtypes: str(2)
memory usage: 1.1 MB


### Let's merge the data with a left join

In [415]:
patients

,patient_id,date_of_birth,patient_sex,dob_age,age_bracket
0,PT00001,1956-07-22,Male,68,65+
1,PT00002,1982-11-05,Female,42,40-64
2,PT00003,1978-02-14,Male,46,40-64
3,PT00004,2004-09-02,Female,20,18-39
4,PT00005,1948-03-19,Male,76,65+
5,PT00006,1988-06-21,Female,36,18-39
6,PT00007,1992-08-15,Female,32,18-39
7,PT00008,1970-12-01,Male,54,40-64
8,PT00009,1960-04-28,Female,64,40-64
9,PT00010,1985-05-30,Male,39,18-39


In [416]:
merged_data = patient_visits.merge(icd_codes,how = 'left',left_on='icd_code',right_on='Code')

In [417]:
merged_data

,visit_id,patient_id,visit_date,date_of_birth,patient_age,patient_sex,icd_code,cpt_code,Code,Short Description
0,V000001,PT00001,2022-03-15,1956-07-22,66,Male,I10,12345,I10,Essential (primary) hypertension
1,V000002,PT00002,2022-04-22,1982-11-05,39,Female,E11.9,23456,NaN,NaN
2,V000003,PT00003,2022-05-10,1978-02-14,44,Male,J45.909,34567,NaN,NaN
3,V000004,PT00004,2023-01-18,2004-09-02,57,Female,N39.0,45678,NaN,NaN
4,V000005,PT00005,2023-07-30,1948-03-19,75,Male,I25.110,56789,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
195,V000018,PT00018,2024-09-09,1977-12-03,46,Female,I10,98011,I10,Essential (primary) hypertension
196,V000019,PT00019,2022-05-25,1995-07-21,28,Male,J02.9,21004,NaN,NaN
197,V000020,PT00020,2023-07-12,2004-11-02,38,Female,E78.0,26009,NaN,NaN
198,V000021,PT00021,2024-04-03,1969-04-08,55,Male,I10,52018,I10,Essential (primary) hypertension


### When checking the info, we can see that only 21 rows were populated

In [418]:
merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   visit_id           200 non-null    str           
 1   patient_id         200 non-null    str           
 2   visit_date         200 non-null    datetime64[us]
 3   date_of_birth      200 non-null    datetime64[us]
 4   patient_age        200 non-null    int64         
 5   patient_sex        200 non-null    str           
 6   icd_code           200 non-null    str           
 7   cpt_code           200 non-null    str           
 8   Code               21 non-null     str           
 9   Short Description  21 non-null     str           
dtypes: datetime64[us](2), int64(1), str(7)
memory usage: 15.8 KB


### With a left anti join, we can see that the missing icd codes all had a dot(.) after them. Let's remove the dot and try again

In [419]:
patient_visits.merge(icd_codes,how = 'left_anti',left_on='icd_code',right_on='Code')

,visit_id,patient_id,visit_date,date_of_birth,patient_age,patient_sex,icd_code,cpt_code,Code,Short Description
1,V000002,PT00002,2022-04-22,1982-11-05,39,Female,E11.9,23456,NaN,NaN
2,V000003,PT00003,2022-05-10,1978-02-14,44,Male,J45.909,34567,NaN,NaN
3,V000004,PT00004,2023-01-18,2004-09-02,57,Female,N39.0,45678,NaN,NaN
4,V000005,PT00005,2023-07-30,1948-03-19,75,Male,I25.110,56789,NaN,NaN
5,V000006,PT00006,2023-11-12,1988-06-21,35,Female,F32.9,67890,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
193,V000016,PT00016,2022-01-29,1962-03-29,60,Female,N18.9,94012,NaN,NaN
194,V000017,PT00017,2023-02-14,1989-05-16,34,Female,G43.909,95022,NaN,NaN
196,V000019,PT00019,2022-05-25,1995-07-21,28,Male,J02.9,21004,NaN,NaN
197,V000020,PT00020,2023-07-12,2004-11-02,38,Female,E78.0,26009,NaN,NaN


In [420]:
patient_visits['icd_code_clean'] = patient_visits['icd_code'].str.replace('.','',regex=False)

In [421]:
patient_visits['icd_code_clean']

0         I10
1        E119
2      J45909
3        N390
4      I25110
        ...  
195       I10
196      J029
197      E780
198       I10
199      F411
Name: icd_code_clean, Length: 200, dtype: str

In [422]:
cleaned_merged_data = patient_visits.merge(icd_codes,how = 'left',left_on='icd_code_clean',right_on='Code')

In [423]:
cleaned_merged_data

,visit_id,patient_id,visit_date,date_of_birth,patient_age,patient_sex,icd_code,cpt_code,icd_code_clean,Code,Short Description
0,V000001,PT00001,2022-03-15,1956-07-22,66,Male,I10,12345,I10,I10,Essential (primary) hypertension
1,V000002,PT00002,2022-04-22,1982-11-05,39,Female,E11.9,23456,E119,E119,Type 2 diabetes mellitus without complications
2,V000003,PT00003,2022-05-10,1978-02-14,44,Male,J45.909,34567,J45909,J45909,"Unspecified asthma, uncomplicated"
3,V000004,PT00004,2023-01-18,2004-09-02,57,Female,N39.0,45678,N390,N390,"Urinary tract infection, site not specified"
4,V000005,PT00005,2023-07-30,1948-03-19,75,Male,I25.110,56789,I25110,I25110,Athscl heart disease of native cor art w unsta...
...,...,...,...,...,...,...,...,...,...,...,...
195,V000018,PT00018,2024-09-09,1977-12-03,46,Female,I10,98011,I10,I10,Essential (primary) hypertension
196,V000019,PT00019,2022-05-25,1995-07-21,28,Male,J02.9,21004,J029,J029,"Acute pharyngitis, unspecified"
197,V000020,PT00020,2023-07-12,2004-11-02,38,Female,E78.0,26009,E780,E780,Pure hypercholesterolemia
198,V000021,PT00021,2024-04-03,1969-04-08,55,Male,I10,52018,I10,I10,Essential (primary) hypertension


In [424]:
cleaned_merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   visit_id           200 non-null    str           
 1   patient_id         200 non-null    str           
 2   visit_date         200 non-null    datetime64[us]
 3   date_of_birth      200 non-null    datetime64[us]
 4   patient_age        200 non-null    int64         
 5   patient_sex        200 non-null    str           
 6   icd_code           200 non-null    str           
 7   cpt_code           200 non-null    str           
 8   icd_code_clean     200 non-null    str           
 9   Code               200 non-null    str           
 10  Short Description  200 non-null    str           
dtypes: datetime64[us](2), int64(1), str(8)
memory usage: 17.3 KB


### Now we have all the icd codes mapped

In [425]:
visits_with_desc = cleaned_merged_data.drop(columns = ['Code','patient_age','patient_sex','date_of_birth'])

visits_with_desc = visits_with_desc.rename(columns={'Short Description':'icd_code_description'})

visits_with_desc

,visit_id,patient_id,visit_date,icd_code,cpt_code,icd_code_clean,icd_code_description
0,V000001,PT00001,2022-03-15,I10,12345,I10,Essential (primary) hypertension
1,V000002,PT00002,2022-04-22,E11.9,23456,E119,Type 2 diabetes mellitus without complications
2,V000003,PT00003,2022-05-10,J45.909,34567,J45909,"Unspecified asthma, uncomplicated"
3,V000004,PT00004,2023-01-18,N39.0,45678,N390,"Urinary tract infection, site not specified"
4,V000005,PT00005,2023-07-30,I25.110,56789,I25110,Athscl heart disease of native cor art w unsta...
...,...,...,...,...,...,...,...
195,V000018,PT00018,2024-09-09,I10,98011,I10,Essential (primary) hypertension
196,V000019,PT00019,2022-05-25,J02.9,21004,J029,"Acute pharyngitis, unspecified"
197,V000020,PT00020,2023-07-12,E78.0,26009,E780,Pure hypercholesterolemia
198,V000021,PT00021,2024-04-03,I10,52018,I10,Essential (primary) hypertension


In [426]:
final_patient_visits = visits_with_desc.merge(patients, on='patient_id',how = 'left')

In [427]:
final_patient_visits.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   visit_id              200 non-null    str           
 1   patient_id            200 non-null    str           
 2   visit_date            200 non-null    datetime64[us]
 3   icd_code              200 non-null    str           
 4   cpt_code              200 non-null    str           
 5   icd_code_clean        200 non-null    str           
 6   icd_code_description  200 non-null    str           
 7   date_of_birth         200 non-null    datetime64[us]
 8   patient_sex           200 non-null    str           
 9   dob_age               200 non-null    int32         
 10  age_bracket           200 non-null    str           
dtypes: datetime64[us](2), int32(1), str(8)
memory usage: 16.5 KB


In [428]:
final_patient_visits.to_csv(r"D:\Google Drive\Data Analysis Education\Portfolio Projects\Diagnosing the Data An Analysis of Patient Demographics, Common Conditions, and Care Utilization\Exports\CSV\final_patient_visits.csv",index=False)

In [429]:
pd.read_csv(r"D:\Google Drive\Data Analysis Education\Portfolio Projects\Diagnosing the Data An Analysis of Patient Demographics, Common Conditions, and Care Utilization\Exports\CSV\final_patient_visits.csv").shape

(200, 11)

In [430]:
pd.read_csv(r"D:\Google Drive\Data Analysis Education\Portfolio Projects\Diagnosing the Data An Analysis of Patient Demographics, Common Conditions, and Care Utilization\Exports\CSV\final_patient_visits.csv").columns

Index(['visit_id', 'patient_id', 'visit_date', 'icd_code', 'cpt_code',
       'icd_code_clean', 'icd_code_description', 'date_of_birth',
       'patient_sex', 'dob_age', 'age_bracket'],
      dtype='str')